In [1]:
%pip install -q pandas pyarrow

import csv
from pathlib import Path

import pandas as pd

DATA = Path("data")  # data/ folder next to the notebook
DATA.mkdir(exist_ok=True)

In [2]:
# ORCAS click logs (~330 MB) and MS MARCO documents (~8.4 GB) go to data/; -C - resumes a partial download.
!curl -fL -C - -o "{DATA}/orcas.tsv.gz" https://msmarco.z22.web.core.windows.net/msmarcoranking/orcas.tsv.gz
!curl -fL -C - -o "{DATA}/msmarco-docs.tsv.gz" https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-docs.tsv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  314M  100  314M    0     0   9.8M      0  0:00:32  0:00:32 --:--:--  9.8M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 8054M  100 8054M    0     0  8914k      0  0:15:25  0:15:25 --:--:-- 9983k


In [3]:
# pandas reads .gz directly. QUOTE_NONE and keep_default_na stop stray quotes and queries like "null" from being misread.
# Arrow-backed strings use a fraction of the RAM of Python string objects.
orcas = pd.read_csv(
    DATA / "orcas.tsv.gz",
    sep="\t",
    names=["qid", "query", "did", "url"],
    dtype="string[pyarrow]",
    quoting=csv.QUOTE_NONE,
    keep_default_na=False,
)
print(orcas.shape)
orcas.head()

(18823602, 4)


,qid,query,did,url
0,9265503,github,D1265400,https://desktop.github.com/
1,9265503,github,D3438005,https://github.com/
2,6832981,youtube,D2923232,https://au.youtube.com/
3,9571352,!,D1238374,https://www.englishclub.com/writing/punctuatio...
4,9571352,!,D889000,http://nationalpunctuationday.com/exclamationp...


In [4]:
# Read the 8 GB file in chunks and keep only titles of clicked docs, so the full file is never in memory.
clicked = orcas["did"].unique()
chunks = pd.read_csv(
    DATA / "msmarco-docs.tsv.gz",
    sep="\t",
    names=["did", "url", "title", "body"],
    usecols=["did", "title"],
    dtype="string[pyarrow]",
    quoting=csv.QUOTE_NONE,
    keep_default_na=False,
    chunksize=100_000,
)
docs = pd.concat(chunk[chunk["did"].isin(clicked)] for chunk in chunks)
print(docs.shape)
docs.head()

(1422029, 2)


,did,title
6,D1150618,Netting the Highest Quality Safety Solutions
9,D3048094,"I have trouble swallowing due to MS, can I cru..."
14,D2883971,Urinary Incontinence
16,D1281784,What is Subrogation and why is it important?
18,D560769,CSF Analysis


In [5]:
# Each click maps to one document, so validate many-to-one.
df = orcas.merge(docs, on="did", validate="many_to_one")
del orcas, docs  # free the inputs before the next steps
print(len(df))

18823602


In [6]:
# Look at the word-count distribution before picking a cap.
df["query"] = df["query"].str.strip()
df["title"] = df["title"].str.strip()

# str.count gives word counts directly; str.split would build a list for every row.
query_words = df["query"].str.count(r"\S+")
title_words = df["title"].str.count(r"\S+")

print(query_words.describe())
print(title_words.describe())

count    18823602.0
mean       3.250992
std         1.59788
min             0.0
25%             2.0
50%             3.0
75%             4.0
max           100.0
Name: query, dtype: Float64
count    18823602.0
mean       4.612314
std       15.380431
min             0.0
25%             2.0
50%             3.0
75%             6.0
max         37157.0
Name: title, dtype: Float64


In [9]:
# Set this from the describe() output above: just past where the long tail starts.
MAX_WORDS = 30

# [^\W_] matches any letter or digit.
has_text = df["query"].str.contains(r"[^\W_]") & df["title"].str.contains(r"[^\W_]")
short_enough = (query_words <= MAX_WORDS) & (title_words <= MAX_WORDS)

df = df[has_text & short_enough].reset_index(drop=True)
del query_words, title_words
print(len(df))
df.head()

16627536


,qid,query,did,url,title
0,9265503,github,D1265400,https://desktop.github.com/,The new native
1,9265503,github,D3438005,https://github.com/,Built for developers
2,4896888,! c,D579719,https://en.wikipedia.org/wiki/C_%28programming...,C (programming language)
3,10001890,! c++,D1094795,https://en.wikipedia.org/wiki/Operators_in_C_a...,Operators in C and C++
4,12449286,! calculator,D1962946,https://web2.0calc.com/,Scientific Calculator


In [11]:
# One clicked page per query, so each question has a single right answer.
df = df.drop_duplicates("qid").reset_index(drop=True)

# Non-ASCII queries are mostly non-English, and Qwen tends to translate them instead of rewriting.
df = df[~df["query"].str.contains(r"[^\x00-\x7F]")]

# 150k training pairs plus 2,500 held-out queries (2,000 test, 500 dev), set aside before any rewriting.
df = df.sample(n=152_500, random_state=0).reset_index(drop=True)
df["split"] = "train"
df.loc[:1999, "split"] = "test"
df.loc[2000:2499, "split"] = "dev"
print(df["split"].value_counts())

split
train    150000
test       2000
dev         500
Name: count, dtype: int64


In [ ]:
# Qwen2.5-7B-Instruct in 4-bit fits on the T4. T4s lack bfloat16, so compute in float16.
%pip install -q transformers accelerate bitsandbytes
import random, time
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
SYSTEM = """You rewrite search-engine queries into questions a person would ask an AI assistant that searches their own browsing history.

You will get the keyword query someone typed and the title of the page they clicked. Imagine that same person days later, trying to find that page again by asking the assistant in plain, everyday words.

Rules:
- Write one question or request, 6 to 20 words, in casual language.
- Refer to the page as something they already saw ("that article", "the site I used", "where I read about"), and vary how you do it.
- Keep the main name a person would remember, such as the site, show, brand or place, so the page can still be found. Describe the rest loosely, the way people do when half-remembering.
- Never copy three or more words in a row from the page title.
- Use only information in the query or the title. Do not invent details like dates, times, prices or reasons.
- Output only the question, with no quotes or explanation."""

# Each prompt shows 3 of these 9, chosen by qid, so rewrites don't all copy the same phrasing.
EXAMPLES = [
    ("stranger things netflix", "Stranger Things | Netflix Official Site",
     "What was that show I was watching on Netflix?"),
    ("chicken thigh oven temp", "How Long to Bake Chicken Thighs at 400°F",
     "What temperature did that recipe say to bake chicken thighs at?"),
    ("irs w4 form", "About Form W-4, Employee's Withholding Certificate | Internal Revenue Service",
     "Where's the W-4 tax form I was filling out?"),
    ("how to tie a tie youtube", "How to Tie a Tie (Windsor Knot) - YouTube",
     "Find that YouTube video I watched on tying a tie."),
    ("list comprehension python", "Python - List Comprehension - GeeksforGeeks",
     "Find that Python tutorial I was reading on list comprehensions."),
    ("how many cups in a quart", "How Many Cups in a Quart? | Kitchen Conversions",
     "Find the page I used to convert quarts to cups."),
    ("symptoms of low iron", "Iron Deficiency Anemia: Symptoms, Causes & Treatment",
     "What was that article I read about signs of low iron?"),
    ("best budget laptops", "The 6 Best Budget Laptops - RTINGS.com",
     "Which cheap laptops was I looking at?"),
    ("zillow austin rentals", "Austin TX Apartments for Rent | Zillow",
     "Pull up those Austin apartments I was looking at on Zillow."),
]

def build_messages(qid, query, title):
    shots = random.Random(qid).sample(EXAMPLES, 3)
    messages = [{"role": "system", "content": SYSTEM}]
    for q, t, answer in shots:
        messages.append({"role": "user", "content": f"Query: {q}\nPage title: {t}"})
        messages.append({"role": "assistant", "content": answer})
    messages.append({"role": "user", "content": f"Query: {query}\nPage title: {title}"})
    return messages

In [ ]:
# Greedy decoding; score = average log-probability of the generated tokens (higher = Qwen was more confident).
@torch.no_grad()
def rewrite_batch(rows):
    prompts = [
        tokenizer.apply_chat_template(build_messages(r.qid, r.query, r.title), tokenize=False, add_generation_prompt=True)
        for r in rows
    ]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**inputs, do_sample=False, max_new_tokens=40, output_scores=True, return_dict_in_generate=True)

    new_tokens = out.sequences[:, inputs["input_ids"].shape[1]:]
    logprobs = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
    mask = new_tokens != tokenizer.pad_token_id  # ignore padding after a rewrite ends
    scores = torch.where(mask, logprobs, 0).sum(1) / mask.sum(1)

    texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return [t.strip() for t in texts], scores.tolist()

# Time 500 first (as the proposal plans), then set N_REWRITE = 50_000.
N_REWRITE = 500
BATCH = 32
to_rewrite = df[df["split"] == "train"].head(N_REWRITE)

texts, scores = [], []
start = time.time()
for i in tqdm(range(0, len(to_rewrite), BATCH)):
    t, s = rewrite_batch(list(to_rewrite.iloc[i : i + BATCH].itertuples()))
    texts += t
    scores += s
print(f"{len(texts)} rewrites in {time.time() - start:.0f}s")

# New columns only; query is untouched. Rows not rewritten stay empty.
df["rewrite"] = pd.Series(texts, index=to_rewrite.index, dtype="string")
df["score"] = pd.Series(scores, index=to_rewrite.index)
df.loc[to_rewrite.index, ["query", "title", "rewrite", "score"]].head(10)

In [ ]:
# Save the final table to Google Drive so other notebooks can load it.
from google.colab import drive
drive.mount("/content/drive")

OUT = Path("/content/drive/MyDrive/Conversational-History")
OUT.mkdir(parents=True, exist_ok=True)  # creates the folder only if it's missing

df.to_parquet(OUT / "orcas_rewrites.parquet", index=False)
print(f"Saved {len(df):,} rows to {OUT / 'orcas_rewrites.parquet'}")

# To load in another notebook:
#   from google.colab import drive; drive.mount("/content/drive")
#   df = pd.read_parquet("/content/drive/MyDrive/Conversational-History/orcas_rewrites.parquet")